In [1]:
from unstructured.chunking.title import chunk_by_title
from unstructured.staging.base import dict_to_elements
from unstructured.partition.pdf import partition_pdf
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document
from langchain_openai import AzureOpenAIEmbeddings
import os
import glob
import re


/Users/sivahari/miniconda3/envs/wahl311/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from defaults import default_env_vars

In [3]:
files = glob.glob("schedule/*.pdf")

documents = []
doccount = 0
print("Start Processing")
for f in files:
    print("Processing doc ", doccount)
    doccount += 1
    try:
        elements = partition_pdf(f, strategy="hi_res", hi_res_model_name="yolox", df_infer_table_structure=True)
        chunked_elements = chunk_by_title(elements)
        
        for element in chunked_elements:
            metadata = element.metadata.to_dict()
            metadata['languages'] = metadata['languages'][0]
            documents.append(Document(page_content=element.text, metadata=metadata))
    except Exception as e:
        print(f"Error in {f}, {e}")
embeddings = AzureOpenAIEmbeddings(
      model = 'dpa-project-text-embedding-3-large',
      azure_endpoint = default_env_vars["EMBEDEP"],
      api_key=default_env_vars["EMBEDKEY"],
      api_version='2023-05-15'
)


vectorstore = Chroma.from_documents(documents, embeddings)
print("All documents processed")

Start Processing
Processing doc  0
Processing doc  1
All documents processed


In [8]:
print(len(documents))

22


In [9]:
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.chains import create_retrieval_chain
from langchain_openai import AzureChatOpenAI
from langchain.prompts import ChatPromptTemplate


In [10]:
llm = AzureChatOpenAI(temperature=0.0, 
                       model='dpa-project-gpt4o', 
                       api_key=default_env_vars["CHATGPTKEY"], 
                       azure_endpoint=default_env_vars["CHATGPTEP"], 
                       api_version='2024-08-01-preview')

In [11]:
retriever = vectorstore.as_retriever()
system_prompt = (
        "You are an assistant for question-answering tasks about an Faculty Development Program. "
        "Use the following pieces of retrieved context to answer "
        "the question. If you don't know the answer, say that you "
        "don't know."
        "\n\n"
        "{context}"
)
prompt = ChatPromptTemplate.from_messages(
        [
            ("system", system_prompt),
            ("human", "{input}"),
        ]
)
question_answer_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)
result = rag_chain.invoke({"input": 'what is this program'})

In [12]:
result['answer']

'The program is an online 6-day Faculty Development Programme (FDP) scheduled for 2024-25, focusing on the thrust area of Advanced Computing, specifically in Artificial Intelligence (AI). The FDP is titled "Advanced Techniques in AI and Machine Learning" and will take place from January 6, 2025, to January 11, 2025. It is organized by the AICTE Training and Learning (ATAL) Academy and aims to enhance the knowledge and skills of participants in AI and machine learning techniques. The program includes various sessions covering topics such as Artificial Neural Networks, Transformer Models, Generative Adversarial Networks, Recurrent Neural Networks, and Large Language Models.'

In [14]:
result["context"]

[Document(metadata={'file_directory': 'schedule', 'filename': 'Atal Schedule 2024-25 Final.pdf', 'filetype': 'application/pdf', 'languages': 'eng', 'last_modified': '2025-01-07T13:48:00', 'orig_elements': 'eJzdVE1r3DAU/CvCpxYSV9+y97Y0BAIpDXRvSzCy9LwrsGXXltMsof+9krMbCl1KCOSS48x7Y+tp5mn7lEELHfhQOZutUNYQSWrFaytLJaTFjJra0qYg0GDCmzK7QFkHQVsddOx/ykzfj9Z5HWBacKsP/RyqPbjdPkSGKIyj5kj/cjbsI0vpwg698yHptlsmRF5cIEJlXtxfoBOmpMxpwkQRkuMzxLMiMtl0mAJ0aYo79wjtj0EbyH7HgoUAJrjeV6bV01QNY1/HNpzTgsRy41qorBtjTz8ekn4ye7BzC9mx6nUHiV8H3aIfxyKimPJLKtB1HL/NB9uc2sNhWNr1MLTO6PTnL8dyq/1u1rvlsrYZ+F12v7BTqLreusbBYkP8tLjE5BKrDWErXqwwTuohKis/dzWM6WrTbAEe0zVn333rPCCJrvQBXWszt+GAruAB2n5I/qK7sd+Nuutgejn4yyTXV3dosx/nKaD1CHqF1vZBewMWfe27YQ7O79Cn9c3ndIbTcBsX2uV6/4mQxSXDWHKtmwJ4WSpqtbACaiUYs+TdIkQLvASCFTlOCTlhKZ8xK4uzeOn/b4A+RELWm/XtqwzUuLFGYqU5h7rgrLCNFio+CHWtmJL83QwUQuU8rjhluUgGnbAkuVpWnpciV+eIRfG2N4BhitVH8XhZ5OTrXyu8AbP37uccF995tL5B2lv0TZt9ei9uQY8+rvergsFxXchCNaZssCWmFqKuBTNNWZJGWfx+wVCqXIKgcF4m34+YYZnzJQdUyZydIZ4VbwuGVIp/mF

In [15]:
result = rag_chain.invoke({"input": 'what is the schedule?'})

In [16]:
print(result['answer'])

The Faculty Development Program on "Advanced Techniques in AI and Machine Learning" is scheduled from January 6th to January 11th, 2025. The sessions are held online in the evening from 6:00 PM to 9:00 PM, with an extended schedule on Saturday from 2:00 PM to 8:00 PM. Here is the detailed schedule:

- **Day 1 (06/01/2025):** 
  - 6:00 PM to 7:30 PM: Inaugural Session

- **Day 2 (07/01/2025):** 
  - 6:00 PM to 7:30 PM: Session 3

- **Day 3 (08/01/2025):** 
  - 6:00 PM to 7:30 PM: Session 5
  - Topic: Artificial Neural Networks and Deep Learning

- **Day 4 (09/01/2025):** 
  - 6:00 PM to 7:30 PM: Session 7
  - Topic: Transformer Models

- **Day 5 (10/01/2025):** 
  - 2:00 PM to 3:30 PM: Session 9
  - Topic: Generative Adversarial Networks
  - 6:00 PM to 6:30 PM: Session 11
  - Topic: Recurrent Neural Networks

- **Day 6 (11/01/2025):** 
  - 5:00 PM to 6:30 PM: Session 13
  - Topic: LLM fine tuning
  - 6:30 PM to 7:30 PM: Online test & feedback

Please note that the sessions are conducted

In [17]:
result = rag_chain.invoke({"input": 'What will be Ditty Mathew talking about?'})

In [18]:
print(result['answer'])

Dr. Ditty Mathew will be talking about "LLM fine tuning" in Session 13.


In [19]:
result = rag_chain.invoke({"input": 'Should I pay for attending this program?'})

In [20]:
print(result['answer'])

No, there is no charge for registration, course, and certification for attending the workshop.


In [21]:
result = rag_chain.invoke({"input": 'tell me 3 sessions that i should attend to learn basics od LLM'})

In [22]:
print(result['answer'])

To learn the basics of Large Language Models (LLM), you should consider attending the following sessions:

1. Session 11: Topic: Large Language Models
2. Session 8: Topic: Transformer Models
3. Session 6: Topic: Deep Learning for NLP

These sessions cover foundational topics related to LLMs, including transformer models and deep learning techniques used in natural language processing.


In [23]:
result = rag_chain.invoke({"input": 'What will be Sivahari talking about?'})

In [24]:
print(result['answer'])

Mr. Sivahari Nankumar will be talking about "RAG Systems, How to Build a RAG system" in his session.


In [25]:
result = rag_chain.invoke({"input": 'Who are the host of the programm?'})

In [26]:
print(result['answer'])

The program is hosted by experts from academia, industry, and research who handle the theory and hands-on training sessions. Some of the resource persons mentioned include Mr. Joy Sebastian, Dr. Bindu Krishnan, Dr. Ditty Mathew, Mr. Sivahari Nankumar, and Dr. Jeena Kleenankandy.


In [27]:
result = rag_chain.invoke({"input": 'What will be Sivahari talking about?'})

In [28]:
print(result['answer'])

Mr. Sivahari Nankumar will be talking about "RAG Systems, How to Build a RAG system" in his session.


In [29]:
result = rag_chain.invoke({"input": 'When is the session?'})

In [30]:
print(result['answer'])

The Faculty Development Program sessions are scheduled as follows:

- Day 1 (06/01/2025): 6:00PM to 7:30PM
- Day 2 (07/01/2025): 6:00PM to 7:30PM
- Day 3 (08/01/2025): 6:00PM to 7:30PM
- Day 4 (09/01/2025): 6:00PM to 7:30PM
- Day 5 (10/01/2025): 6:00PM to 7:30PM
- Day 6 (11/01/2025): 2:00PM to 8:00PM

The sessions are held in the evening from 6:00PM to 9:00PM, except on Saturday, when they are from 2:00PM to 8:00PM.


### Lets Include chat history

In [32]:
history = []
question1 = 'What will be Sivahari talking about?'
result = rag_chain.invoke({"input": question1})
history.append(['User : ' + question1, 'chatbot : ' + result['answer']])
print(result['answer'])

Mr. Sivahari Nankumar will be talking about "RAG Systems, How to Build a RAG system" in his session.


In [33]:
history

[['User : What will be Sivahari talking about?',
  'chatbot : Mr. Sivahari Nankumar will be talking about "RAG Systems, How to Build a RAG system" in his session.']]

In [34]:
new_question = "When is the session?"

In [35]:
prompt_text = """
Following is the history of the pevious QA system : 
{history}

Following is the new question asked, could be a followup question:
{question}

Give me a standalone question :
"""

In [36]:
from langchain.prompts import ChatPromptTemplate


In [37]:
prompt_template = ChatPromptTemplate.from_template(prompt_text)
messages = prompt_template.format_messages(history=history, 
                                  question=new_question)

In [38]:
resp = llm(messages) ##We called the LLM here not the RAG

/var/folders/g4/q3sxtq7n5wj42x7003zqm7t00000gp/T/ipykernel_38116/70103795.py:1: LangChainDeprecationWarning: The method `BaseChatModel.__call__` was deprecated in langchain-core 0.1.7 and will be removed in 1.0. Use :meth:`~invoke` instead.
  resp = llm(messages) ##We called the LLM here not the RAG


In [39]:
print(resp.content)

When is Mr. Sivahari Nankumar's session on "RAG Systems, How to Build a RAG system"?


In [40]:
result = rag_chain.invoke({"input": resp.content})

In [41]:
print(result['answer'])

Mr. Sivahari Nankumar's session on "RAG Systems, How to Build a RAG system" is scheduled from 8:00 PM to 9:30 PM. However, the specific day of the session is not provided in the context.


In [42]:
history.append(['User : ' + new_question, 'chatbot : ' + result['answer']])

In [43]:
history

[['User : What will be Sivahari talking about?',
  'chatbot : Mr. Sivahari Nankumar will be talking about "RAG Systems, How to Build a RAG system" in his session.'],
 ['User : When is the session?',
  'chatbot : Mr. Sivahari Nankumar\'s session on "RAG Systems, How to Build a RAG system" is scheduled from 8:00 PM to 9:30 PM. However, the specific day of the session is not provided in the context.']]

In [49]:
new_new_question = 'What is after that?'

In [50]:
messages = prompt_template.format_messages(history=history, 
                                  question=new_new_question)

In [51]:
resp = llm(messages) ##We called the LLM here not the RAG

In [52]:
result = rag_chain.invoke({"input": resp.content})

In [53]:
print(result['answer'])

After Mr. Sivahari Nankumar's session on "RAG Systems, How to Build a RAG system," which is scheduled from 8:00PM to 9:30PM, there is no specific session mentioned immediately after it. However, the context does mention an online test and feedback session from 6:30PM to 7:30PM, and a valedictory session, but their exact timing in relation to Mr. Sivahari Nankumar's session is not specified.


In [56]:
result = rag_chain.invoke({"input": "Elias sir have 3 apples, I have 4. How much the total apples costs?"})

In [57]:
print(result['answer'])

I'm sorry, I don't have enough information to determine the total cost of the apples. Could you provide the price per apple?


### Agentic RAG

In [58]:
from langchain_core.tools import tool

In [59]:
from llama_index.core import Settings
from llama_index.llms.azure_openai import AzureOpenAI as AI

In [60]:
from llama_index.core.tools import FunctionTool

def add(x: int, y: int) -> int:
    """Adds two integers together."""
    return x + y

def rag(question: str) -> str: 
    """QA system for a LLM workshop.
    This functions answers questions about the schedule of the workshop,
    details about the speakers in the workshop etc."""
    re_result = rag_chain.invoke({"input": question})
    return re_result['answer']


add_tool = FunctionTool.from_defaults(fn=add)
rag_tool = FunctionTool.from_defaults(fn=rag)


In [61]:
agent = AI( model="gpt-4o",
    deployment_name="dpa-project-gpt4o",
    api_key=default_env_vars["CHATGPTKEY"],
    azure_endpoint=default_env_vars["CHATGPTEP"],
    api_version='2024-08-01-preview',)



response = agent.predict_and_call(
    [add_tool, rag_tool], 
    "what is 2 + 2?", 
    verbose=True
)
print(str(response))

=== Calling Function ===
Calling function: add with args: {"x": 2, "y": 2}
=== Function Output ===
4
4


In [62]:
response = agent.predict_and_call(
    [add_tool, rag_tool], 
    "what is the schedule?", 
    verbose=True
)
print(str(response))

=== Calling Function ===
Calling function: rag with args: {"question": "What is the schedule?"}
=== Function Output ===
The Faculty Development Program is scheduled from January 6th to January 11th, 2025. The sessions are held in the evening from 6:00 PM to 9:00 PM, except on Saturday, when they are from 2:00 PM to 8:00 PM. Here is the schedule:

- Day 1 (06/01/2025): 6:00 PM to 7:30 PM - Inaugural Session
- Day 2 (07/01/2025): 6:00 PM to 7:30 PM - Session 3
- Day 3 (08/01/2025): 6:00 PM to 7:30 PM - Session 5, Topic: Transformer
- Day 4 (09/01/2025): 6:00 PM to 7:30 PM - Session 7, Topic: Artificial Neural Networks and Deep Learning
- Day 5 (10/01/2025): 2:00 PM to 3:30 PM - Session 9, Topic: Generative Adversarial Networks
- Day 6 (11/01/2025): 5:00 PM to 6:30 PM - Session 13, Topic: LLM fine tuning, Name of the Expert: Dr. Ditty Mathew; 6:30 PM to 7:30 PM - Online test & feedback; 7:30 PM to 8:00 PM - Closing session

Please note that the schedule includes various topics related to 

In [63]:
response = agent.predict_and_call(
    [add_tool, rag_tool], 
    "I have 3 apples, my wife have 4 apples. How many apples we together have?", 
    verbose=True
)
print(str(response))

=== Calling Function ===
Calling function: add with args: {"x": 3, "y": 4}
=== Function Output ===
7
7
